# 🎭 Free Talking Photo & Lip-Sync Video Studio
### ১০০% ফ্রি গুগল কোলাব GPU ব্যবহার করে যেকোনো ছবি ও অডিও থেকে নিখুঁত লিপ-সিঙ্ক ভিডিও তৈরি করুন

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaalikolom-ux/video-project/blob/main/colab_talking_photo_lipsync.ipynb)

---
### 📌 ব্যবহারের নিয়ম (একদম সহজ ৩টি ধাপ):
1. উপরে মেনু থেকে **Runtime ➔ Run all** (অথবা কীবোর্ডে `Ctrl + F9`) চাপুন।
2. স্বয়ংক্রিয়ভাবে মডেল ডাউনলোড হবে, আপনার ছবি ও কবিতা লোড হবে এবং নিখুঁত লিপ-সিঙ্ক ভিডিও রেন্ডার হবে।
3. ভিডিও তৈরি শেষ হলে নিচে ভিডিওটি প্লে হবে এবং স্বয়ংক্রিয়ভাবে আপনার কম্পিউটারে ডাউনলোড হয়ে যাবে।

## ⚙️ ধাপ ১: পরিবেশ ও এআই মডেল সেটআপ (Setup Environment & GPU)

In [ ]:
# Check GPU availability
!nvidia-smi

# Clone SadTalker repository
!rm -rf /content/SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
%cd /content/SadTalker

# Install compatible NumPy 1.x & dependencies (fixes NumPy 2.0 compatibility issues in Colab)
!pip install -q "numpy<2.0.0" "numba<0.60"
!pip install -q -r requirements.txt
!pip install -q facexlib gfpgan yacs

# Apply Colab compatibility patches
!find /content/SadTalker -name "*.py" -exec sed -i 's/np.VisibleDeprecationWarning/getattr(np, "VisibleDeprecationWarning", UserWarning)/g' {} + 2>/dev/null || true
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' $(python -c "import site; print(site.getsitepackages()[0])")/basicsr/data/degradations.py 2>/dev/null || true

# Download pre-trained weights
!mkdir -p checkpoints
!mkdir -p gfpgan/weights
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -O ./checkpoints/mapping_00109-model.pth.tar
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -O ./checkpoints/mapping_00229-model.pth.tar
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -O ./checkpoints/SadTalker_V0.0.2_256.safetensors
!wget -nc https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors -O ./checkpoints/SadTalker_V0.0.2_512.safetensors
!wget -nc https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -O ./gfpgan/weights/alignment_WFLW_4HG.pth
!wget -nc https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -O ./gfpgan/weights/detection_Resnet50_Final.pth
!wget -nc https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -O ./gfpgan/weights/GFPGANv1.4.pth
!wget -nc https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth -O ./gfpgan/weights/parsing_parsenet.pth

print("\n✅ মডেল ও লাইব্রেরি সফলভাবে সেটআপ হয়েছে!")

## 📸 ধাপ ২: ছবি ও অডিও লোড করুন (Load Image & Poem Audio)
নিচের কোডটি রান করলে আপনার GitHub রিপোজিটরি থেকে আজকের কবিতা ও ছবিটি সরাসরি লোড হয়ে যাবে।

In [ ]:
import os
import urllib.request

# Auto-fetch your current poem portrait & 66s recitation audio
img_url = "https://raw.githubusercontent.com/kaalikolom-ux/video-project/main/assets/poem_portrait.jpg"
audio_url = "https://raw.githubusercontent.com/kaalikolom-ux/video-project/main/assets/poem_recitation.mp3"

urllib.request.urlretrieve(img_url, "/content/source_image.jpg")
urllib.request.urlretrieve(audio_url, "/content/driven_audio.mp3")

# Convert audio to clean 16kHz WAV for flawless lip-sync processing
!ffmpeg -y -i /content/driven_audio.mp3 -ar 16000 /content/driven_audio.wav 2>/dev/null

print("✅ আপনার ছবি (source_image.jpg) এবং কবিতার আবৃত্তি (driven_audio.wav) সফলভাবে লোড হয়েছে!")

## 🎬 ধাপ ৩: সম্পূর্ণ লিপ-সিঙ্ক ভিডিও তৈরি ও ডাউনলোড (Render & Auto-Download)
এই সেলটি রান করুন। এটি অডিওর প্রতিটি শব্দের সাথে ঠোঁটের নড়াচড়া (Lip-Sync), চোখের পলক ও মুখের অভিব্যক্তি সিঙ্ক করে সম্পূর্ণ এইচডি ভিডিও তৈরি করবে এবং ভিডিওটি আপনার কম্পিউটারে ডাউনলোড করে দেবে।

In [ ]:
import os
import glob
from google.colab import files
from IPython.display import HTML
from base64 import b64encode

%cd /content/SadTalker

# Defensive patches right before inference
!pip install -q "numpy<2.0.0" "numba<0.60" 2>/dev/null || true
!find /content/SadTalker -name "*.py" -exec sed -i 's/np.VisibleDeprecationWarning/getattr(np, "VisibleDeprecationWarning", UserWarning)/g' {} + 2>/dev/null || true
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' $(python -c "import site; print(site.getsitepackages()[0])")/basicsr/data/degradations.py 2>/dev/null || true

!rm -rf /content/results
!mkdir -p /content/results

print("🚀 এআই লিপ-সিঙ্ক ভিডিও তৈরি শুরু হচ্ছে... (অনুগ্রহ করে কিছুক্ষণ অপেক্ষা করুন)")

# Run inference with face enhancer (GFPGAN) for high-definition quality
!python inference.py \
    --driven_audio /content/driven_audio.wav \
    --source_image /content/source_image.jpg \
    --result_dir /content/results \
    --still \
    --preprocess full \
    --enhancer gfpgan

# Find generated video file (recursive search inside /content/results)
video_files = glob.glob("/content/results/**/*.mp4", recursive=True)
if not video_files:
    video_files = glob.glob("/content/results/*.mp4")

if video_files:
    # Prefer enhanced video if available
    enhanced_files = [f for f in video_files if "_enhanced" in f]
    latest_video = max(enhanced_files, key=os.path.getctime) if enhanced_files else max(video_files, key=os.path.getctime)
    
    print(f"\n🎉 ভিডিও সফলভাবে তৈরি হয়েছে: {latest_video}")
    
    # Preview video in notebook
    mp4 = open(latest_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
    <video width=480 controls autoplay>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    
    # Trigger auto-download to user's PC
    print("⬇️ কম্পিউটারে ডাউনলোড শুরু হচ্ছে...")
    files.download(latest_video)
else:
    print("❌ কোনো ভিডিও খুঁজে পাওয়া যায়নি। দয়া করে এরর লগ চেক করুন।")